# X4: Massive black-hole binaries & MCMC (with fast likelihoods)

**Development-stack exercise notebook (LATW `dev` branch).** There is no
Colab button: these exercises target the *development* versions of the LISA
Analysis Tools packages. Set the environment up by cloning LISAanalysistools
and running its installer (it lays every sibling repo out side by side and
editable-installs the development branches):

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

For the workshop on the **pip-released** packages, use the
[`main` branch](https://github.com/lisa-analysis-tools/LATW/tree/main) instead
(branch policy: `main` &harr; pip releases, `dev` &harr; the `install.sh` stack).

In [ ]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

# Eryn's internals still import a legacy prior module (harmless) and the stock
# noise models divide by f = 0 on full grids; silence both so output stays clean.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

This notebook follows a **massive black-hole binary (MBHB)** from a raw
waveform all the way to a **posterior** &mdash; the same journey
[`X6`](X6_GalacticBinaryMCMC.ipynb) made for a galactic binary and
[`X2`](X2_EMRIResponseMCMC.ipynb) for an EMRI, now for LISA's *loudest* source.
An MBHB is the merger of two ~10&#8308;&ndash;10&#8311; M&#8857; black holes at the
centres of colliding galaxies; its full **inspiral&ndash;merger&ndash;ringdown**
(IMR) sweeps through the LISA band in its final days, reaching signal-to-noise
ratios in the **hundreds to thousands**. We will

1. **generate** a short, merger-centred MBHB with the stock, global-fit-aligned
   `PhenomTHMTDIWaveform` and look at its **XYZ TDI** channels (the merger burst);
2. **inject** it and score its **SNR** with the `AnalysisContainer` +
   `XYZ2SensitivityMatrix` atom from [`X1`](X1_SensitivitySNR.ipynb);
3. run a small **standalone MBHB MCMC** &mdash; the source-MCMC skeleton from
   [`X6`](X6_GalacticBinaryMCMC.ipynb) with the waveform swapped out; and
4. meet **heterodyning / relative binning**, the fast-likelihood trick that makes
   MBHB (and every long, loud LISA signal) tractable &mdash; the modern update of
   this workshop's old *Tutorial 4*.

The companion informational notebooks are [`06`](../../06_SourceWaveforms.ipynb)
(the MBHB waveform), [`05`](../../05_ResponseAndTDI.ipynb) (the LISA response &
TDI), and [`07`](../../07_ErynSmallToLarge.ipynb) (Eryn). Everything is in the
**XYZ** TDI basis, on the laptop CPU. The MBHB waveform (phentax `IMRPhenomTHM`)
is the **heaviest** of the four LISA source classes to evaluate, so every step is
kept deliberately short &mdash; a merger-centred snippet of a few days, a modest
cadence, and a tiny sampler run.

### How these exercises work

Each exercise is one of two kinds:

- **Task N** &mdash; you *write code* toward a stated goal. In this answer notebook the
  solution cells are filled in; in the generated student notebook they are blanked
  (a whole cell, or just the key solution lines for a fill-in-the-blank). Every
  Task ends with a **Useful documentation:** list pointing at the Sphinx/Eryn API
  docs and the relevant informational-notebook section.
- **Question** (a `### Question` heading) &mdash; a short *discussion* prompt. No code
  required; the answer sketch here is for the group conversation and is removed in
  the student notebook.

The tasks build on each other in order, so run them top to bottom.

### The same source-MCMC skeleton

[`X6`](X6_GalacticBinaryMCMC.ipynb) showed that every source in the LISA global
fit is sampled with the **same five-stage skeleton**. This notebook reuses it for
an MBHB &mdash; only the waveform changes:

- **Stage A &mdash; Data** (Tasks 1&ndash;2): an injected data stream (here a
  *synthetic* MBHB in XYZ on a short, merger-centred grid), scored for SNR.
- **Stage B &mdash; Template generator** (Task 3): a `signal_gen(*params) -> DomainBase`
  closure that builds an MBHB template.
- **Stage C &mdash; Likelihood** (Task 3): an `AnalysisContainer(data, sens, signal_gen)`
  whose `eryn_likelihood_function` is the sampler's log-likelihood.
- **Stage D &mdash; Sample** (Task 3): Eryn priors + a tempered `EnsembleSampler`, a
  short run, a corner plot against the truth.
- **Stage E &mdash; Pay-off** (Task 4): the **heterodyned** fast likelihood that makes
  a real MBHB run affordable.

An MBHB is *the same sampler with a different `signal_gen`* &mdash; but its waveform
is by far the costliest to evaluate, which is why the run here is tiny and
explicitly **not converged**, and why the fast likelihood of Task 4 matters so
much.

## Task 1: Build a short MBHB waveform and plot its XYZ TDI channels (Stage A)

Every analysis starts from a **data stream**. Here we synthesize one: a
single MBHB injected into empty (noiseless) XYZ data, so we know the truth exactly.
The stock, global-fit-aligned waveform is
[`PhenomTHMTDIWaveform`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/main.html)
(`lisatools.sources.bbh`): a phentax `IMRPhenomTHM` intrinsic waveform (higher
modes (2,1),(3,3),(4,4) + the dominant (2,2), negative modes on) put through the
LISA `pyResponseTDI` response &mdash; 2nd-generation TDI, **XYZ** channels, response
order 30. A bare instance reproduces the generator the stock `all_sources` fit uses
for its default MBH branch; only the *run-specific* grid / orbits / `waveform_t0`
are passed in.

Because an MBHB is a genuine **chirp** &mdash; its frequency sweeps up through the
band and ends in a high-amplitude merger and ringdown &mdash; and because a
full-span phentax generation would OOM a laptop, we generate only a **short,
merger-centred window** (here ~4 days, ending just after coalescence). The cell that
builds the generator and the raw TDI channels is **provided**; run it as-is. Your
task is the next cell: **plot** the three XYZ TDI channels and watch the
inspiral&ndash;merger&ndash;ringdown burst.

Useful documentation:
* [`PhenomTHMTDIWaveform`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/main.html) (the stock MBH TDI waveform; see [`06` &sect; Massive black-hole binaries](../../06_SourceWaveforms.ipynb))
* [`TDSignal`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/domains.html#lisatools.domains.TDSignal) /
  [`TDSettings`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/domains.html#lisatools.domains.TDSettings)
* Informational notebooks: see [`06` &sect; Massive black-hole binaries](../../06_SourceWaveforms.ipynb) and [`05` &sect; Response](../../05_ResponseAndTDI.ipynb)

In [ ]:
# imports
from lisatools.sources.bbh import PhenomTHMTDIWaveform
from lisatools.detector import EqualArmlengthOrbits
from lisatools.domains import TDSignal, TDSettings, FDSignal, FDSettings

In [ ]:
# provided ("the data processor"): build the aligned MBHB + its XYZ TDI response.
# Run as-is (the first generation JIT-compiles phentax; ~40 s one-time).
dt = 10.0                                # cadence [s]
Tobs_wf = 4.0 * 86400.0                  # SHORT phentax generation window (~4 days)
Ndata = int(round(Tobs_wf / dt)) + 4096  # data grid (a little pad past merger)

# Aligned defaults are baked in: IMRPhenomTHM higher modes (21, 33, 44) + negative
# modes, 2nd-gen TDI, XYZ, response order 30 -- the all_sources MBH generator. Only
# the run-specific grid / orbits / t0 are passed here.
mbh = PhenomTHMTDIWaveform(
    waveform_t0=0.0,
    data_td_settings=TDSettings(N=Ndata, dt=dt, t0=0.0, force_backend="cpu"),
    Tobs=Tobs_wf,
    sampling_frequency=1.0 / dt,
    orbits=EqualArmlengthOrbits(force_backend="cpu"),
    force_backend="cpu",
)

# Physical parameters, in the generator's direct-call order:
#   (m1, m2, s1z, s2z, dist[Mpc], phi_ref, iota, psi, ra, dec, merger_time)
m1, m2 = 1.2e6, 0.8e6                     # component masses [solar masses]
s1z, s2z = 0.3, 0.2                       # aligned dimensionless spins
dist, phi_ref, iota, psi = 15000.0, 1.0, 1.0, 1.2   # Mpc, rad, rad, rad
ra, dec = 2.0, 0.4                        # sky position [rad] (ICRS)
merger_time = 0.85 * Tobs_wf             # place the merger near the end of the window

times, chans = mbh.compute_tdi_channels(
    m1, m2, s1z, s2z, dist, phi_ref, iota, psi, ra, dec, merger_time)
chans = np.asarray(chans)
if chans.ndim == 3:                       # single-source path may return (1, 3, N)
    chans = chans[0]
times = np.asarray(times)
if times.ndim == 2:
    times = times[0]
N = chans.shape[-1]
print("XYZ TDI shape:", chans.shape, "  window:", round(N * dt / 86400.0, 1), "days")

Now plot the injection (time measured in days from the window start). Over
the *full* window (left) the loud **merger burst** near the end dwarfs everything
else &mdash; almost all of the amplitude arrives at coalescence. **Zoom in** on the
merger (right) and the full **inspiral&ndash;merger&ndash;ringdown** morphology is
unmistakable: the chirp winds up in frequency and amplitude, peaks at merger, and
rings down &mdash; in all three XYZ channels, each modulated differently by the LISA
response.

### Question: the role of the merger

The waveform you just plotted spends most of the window as a slow, quiet
inspiral and then delivers almost all of its amplitude in a brief burst at the end.
Why is that **merger&ndash;ringdown** burst so important for LISA science &mdash;
what does it add beyond the inspiral?

*Discussion.* Two things make the merger decisive. First, **signal-to-noise**: the
strain amplitude grows as the black holes approach and peaks at merger, so a large
fraction of the total SNR &mdash; the hundreds-to-thousands you will measure in
Task 2 &mdash; accumulates in those final hours. That is what lets LISA see MBHBs
across the observable Universe. Second, **information**: the inspiral chirp fixes the
masses and (aligned) spins through its phasing, but the **merger and ringdown**
probe the *strong-field, highly dynamical* regime &mdash; the ringdown frequencies
are set by the final black hole's mass and spin (black-hole spectroscopy), a clean
test of general relativity. The merger is also the moment the signal sweeps fastest
in frequency, and combined with the LISA response's sky modulation over the run it
sharpens the **sky localization** (crucial for finding an electromagnetic
counterpart). An MBHB seen through merger is therefore both the loudest and the most
information-rich single event LISA will record &mdash; the flip side (Task 3) is a
razor-sharp likelihood.

## Task 2: Inject the MBHB and compute its SNR (Stage A, cont.)

Now score the injection's **optimal SNR** with the analysis atom from
[`X1`](X1_SensitivitySNR.ipynb). Take the XYZ time series from Task 1 to the
frequency domain (LDC convention `FD = dt * rfft(td)`), band-limit it to the MBHB's
support with an
[`FDSignal`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/domains.html#lisatools.domains.FDSignal),
pair it with an
[`XYZ2SensitivityMatrix`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/sensitivity.html#lisatools.sensitivity.XYZ2SensitivityMatrix)
(the 3&times;3 XYZ noise covariance), drop both into an
[`AnalysisContainer`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#analysis-container),
and read off `.snr()`. MBHBs are loud &mdash; expect an SNR in the **hundreds to
thousands**.

Useful documentation:
* [`XYZ2SensitivityMatrix`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/sensitivity.html#lisatools.sensitivity.XYZ2SensitivityMatrix) /
  [`AnalysisContainer`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#analysis-container) / [`.snr`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.snr)
* [`FDSignal`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/domains.html#lisatools.domains.FDSignal) /
  [`FDSettings`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/domains.html#lisatools.domains.FDSettings)
* Informational notebooks: see [`06` &sect; Massive black-hole binaries](../../06_SourceWaveforms.ipynb) and [`X1`](X1_SensitivitySNR.ipynb)

In [ ]:
# imports
from lisatools.sensitivity import XYZ2SensitivityMatrix
from lisatools.analysiscontainer import AnalysisContainer

### Question: MBHB parameter measurability

An MBHB with SNR in the thousands yields extraordinarily tight constraints
&mdash; masses to fractions of a percent, sky areas of a few square degrees or
better. Why does an MBHB pin its parameters so much better than, say, a galactic
binary or a quiet stellar-origin binary?

*Discussion.* Parameter precision scales roughly as **1/SNR**, and an MBHB seen
through merger is by far the loudest single LISA source, so it starts with a huge
advantage. But it is not only loudness. (1) **Broadband chirp.** The signal sweeps
across a wide frequency range in a short time, so it samples the whole sensitivity
bucket; the rapidly accumulating phase makes the **masses** (and aligned **spins**)
exquisitely measurable &mdash; a tiny mass error dephases the template over the
chirp and destroys the overlap (you will feel this in Task 3). (2) **The LISA
response over the final days.** As the constellation orbits and cartwheels, its
antenna pattern modulates the loud signal, and the strong frequency sweep at merger
gives many independent projections of the two polarizations into XYZ &mdash; this is
what **localizes the source on the sky** and fixes inclination, distance, and
polarization. (3) **Merger + ringdown.** The high-frequency burst adds SNR and the
ringdown independently constrains the final mass and spin. The combination is why an
MBHB gives the tight sky (for counterpart searches) and mass/spin measurements that
make it a flagship LISA source &mdash; and why its posterior is a tiny, sharply
curved sliver that is genuinely hard to sample.

## Task 3: A standalone MBHB MCMC (Stages B-D)

Now take an MBHB from injected data all the way to a **posterior**, with
exactly the source-MCMC skeleton from [`X6`](X6_GalacticBinaryMCMC.ipynb) &mdash;
only the waveform is different. We sample **three** parameters &mdash; the total mass
$M_T = m_1 + m_2$, the mass ratio $q = m_1/m_2 \ge 1$, and the merger time
$t_\text{merger}$ &mdash; holding the rest fixed at truth.

There is one practical concession. Each likelihood call generates a *full* phentax
IMR waveform and pushes it through the response, which is **the heaviest per-template
cost of any LISA source** (~10&ndash;20 s on a laptop CPU). To keep a tempered run
inside the workshop's time budget, the **provided** cell below builds a *lighter*
`PhenomTHMTDIWaveform` for the sampler: one higher mode ((3,3)) instead of the full
(2,1),(3,3),(4,4), on a shorter 2-day window at a coarser cadence. It is the same
stock class and the same skeleton &mdash; just trimmed so ~30 evaluations finish in a
few minutes (the *real* answer to this cost is Task 4). We free the Task-1 generator
first to keep the memory footprint down.

- **Stage B** &mdash; a `mbh_signal_gen(mT, q, t_merger)` closure that fills the fixed
  parameters, generates the MBHB, and returns a band-limited `FDSignal`.
- **Stage C** &mdash; an `AnalysisContainer(data, sens, signal_gen=...)` whose
  [`eryn_likelihood_function`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.eryn_likelihood_function)
  is the sampler's log-likelihood (`likelihood_source_only=True`, so it reads
  $\approx 0$ at the truth).
- **Stage D** &mdash; Eryn priors, a *tempered*
  [`EnsembleSampler`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler),
  a short run started in a tight ball around the injection, and a corner plot.

Useful documentation:
* [`AnalysisContainer`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#analysis-container) /
  [`eryn_likelihood_function`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.eryn_likelihood_function)
* [`EnsembleSampler`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler) /
  [`State`](https://lisa-analysis-tools.github.io/Eryn/user/state.html#eryn.state.State) /
  [`ProbDistContainer`](https://lisa-analysis-tools.github.io/Eryn/user/prior.html#eryn.prior.ProbDistContainer) /
  [`UniformDistribution`](https://lisa-analysis-tools.github.io/Eryn/user/prior.html)
* Informational notebooks: see [`07` &sect; A fixed-dimension ensemble sampler](../../07_ErynSmallToLarge.ipynb),
  [`X3`](X3_FixedDimMCMC.ipynb), and [`X6`](X6_GalacticBinaryMCMC.ipynb)

In [ ]:
# imports
from lisatools.sources.bbh import MBH_PHENOM_DEFAULT_WAVEFORM_KWARGS
from eryn.ensemble import EnsembleSampler
from eryn.state import State
from eryn.priors import ProbDistContainer, UniformDistribution
from eryn.moves import StretchMove

In [ ]:
# provided ("the data processor"): a lighter MBHB generator + the injected data.
# Run as-is. (Free the Task-1 generator first; then build the sampler's generator.)
del mbh
gc.collect()

dt_mc = 15.0                                    # coarser cadence for the sampler
Tobs_mc = 2.0 * 86400.0                         # shorter 2-day window
Ndata_mc = int(round(Tobs_mc / dt_mc)) + 4096

# one higher mode instead of (21, 33, 44) -> ~2x cheaper per template; still a
# genuine IMR waveform (same stock class, aligned everywhere else).
light_kwargs = dict(MBH_PHENOM_DEFAULT_WAVEFORM_KWARGS)
light_kwargs["higher_modes"] = [33]

mbh_mc = PhenomTHMTDIWaveform(
    waveform_kwargs=light_kwargs,
    waveform_t0=0.0,
    data_td_settings=TDSettings(N=Ndata_mc, dt=dt_mc, t0=0.0, force_backend="cpu"),
    Tobs=Tobs_mc,
    sampling_frequency=1.0 / dt_mc,
    orbits=EqualArmlengthOrbits(force_backend="cpu"),
    force_backend="cpu",
)

# truth (same physical source, on the sampler's grid)
t_merger_mc = 0.85 * Tobs_mc
truth_full = (m1, m2, s1z, s2z, dist, phi_ref, iota, psi, ra, dec, t_merger_mc)

# inject: get_signals_for_residuals places the source on the data grid and returns a
# band-limited FDSignal -- the SAME grid every template will be scored on.
data_mc = mbh_mc.get_signals_for_residuals(*truth_full)
sens_mc = XYZ2SensitivityMatrix(data_mc.settings, model="sangria")
print("data bins:", np.asarray(data_mc.settings.f_arr).size,
      "  injected SNR:", round(float(AnalysisContainer(data_mc, sens_mc).snr())))

**Stage B + C.** Write the template generator `mbh_signal_gen(mT, q, t_merger)`:
map $(M_T, q)$ back to $(m_1, m_2)$ via $m_2 = M_T/(1+q)$, $m_1 = q\,m_2$, fill the
fixed parameters, and return the band-limited `FDSignal` from
`get_signals_for_residuals`. Then build the `AnalysisContainer` likelihood and
confirm it reads $\approx 0$ at the truth &mdash; and that a tiny shift in the merger
time collapses it (the MBHB likelihood is razor-sharp).

In [ ]:
# Stage B: the template generator -- fill the fixed params, return a band-limited FDSignal.
def mbh_signal_gen(mT, q, t_merger):
    """Band-limited XYZ MBHB template in the sampled (mT, q, t_merger) basis."""
    return mbh_mc.get_signals_for_residuals(
        m1_, m2_, s1z, s2z, dist, phi_ref, iota, psi, ra, dec, t_merger)

# Stage C: the likelihood = data + covariance + generator
ac = AnalysisContainer(data_mc, sens_mc, signal_gen=mbh_signal_gen,

mT_true, q_true = m1 + m2, m1 / m2
truth3 = np.array([mT_true, q_true, t_merger_mc])
print(f"logL(truth)            = {ac.eryn_likelihood_function(truth3): .4e}")
# a 1-second shift in the merger time already dephases the loud merger burst
off = truth3 + np.array([0.0, 0.0, 1.0])
print(f"logL(t_merger + 1 s)   = {ac.eryn_likelihood_function(off): .4e}")

**Stage D.** One uniform prior per sampled parameter, a *tempered*
`EnsembleSampler`, and a short run. The priors are **narrow** because a loud MBHB
likelihood is extraordinarily sharp (the 1-second merger shift above already cost
thousands of nats), and the walkers start in a tight ball around the injection
&mdash; a real pipeline gets that seed from a search. We use a small temperature
ladder and the affine-invariant stretch move (`live_dangerously=True` lets us run a
few walkers per temperature at fixed low cost).

> **This chain is deliberately tiny and NOT converged.** With 4 walkers &times;
> 2 temperatures and only 3 recorded steps (~30 template evaluations, chosen to keep
> the laptop run to a few minutes) the marginals are noise, not a posterior &mdash; a
> real MBHB analysis samples all ~11 parameters, runs *far* longer with tuned
> proposals, and checks convergence (autocorrelation time / Gelman&ndash;Rubin; see
> [`X3` &sect; Task 4](X3_FixedDimMCMC.ipynb)). Each likelihood call is a full phentax
> IMR waveform through the response (~10&ndash;20 s here even with one higher mode),
> which is exactly why the fast likelihood of Task 4 exists. We plot it anyway to
> watch the walkers cluster on the truth.

## Task 4: Heterodyning — the fast MBHB likelihood (Stage E)

You just felt the problem: at ~10&ndash;20 s per template, even a toy MBHB run
is minutes long, and a *real* one (all parameters, a full-length loud signal on a
fine grid, millions of frequency bins) is hopeless if every likelihood call rescores
the whole data stream. The production fix &mdash; and the modern update of this
workshop's old *Tutorial 4* &mdash; is **heterodyning**, a.k.a. **relative binning**.

The idea: pick a **reference** template $h_0(f)$ near the current point. For any
nearby trial template $h(f)$, the ratio $r(f) = h(f)/h_0(f)$ is a **slowly varying**
function of frequency (nearby parameters differ mostly by a smooth phase drift), even
though $h$ and $h_0$ themselves oscillate rapidly across millions of bins. So instead
of evaluating the noise-weighted inner products $\langle d\,|\,h\rangle$ and
$\langle h\,|\,h\rangle$ bin-by-bin, you **precompute** summary data
$\langle d\,|\,h_0\rangle$-type coefficients on a *coarse* grid of a few hundred
frequency knots once, then approximate $r(f)$ as piecewise-linear between knots. The
per-likelihood cost drops from $O(N_\text{bins})$ (millions) to
$O(N_\text{knots})$ (hundreds) &mdash; often a **10&ndash;100&times;** speed-up &mdash;
with negligible loss of accuracy as long as the trial stays near the reference (you
refresh $h_0$ when it drifts too far).

BBHx implements exactly this for MBHBs as
[`HeterodynedLikelihood`](https://lisa-analysis-tools.github.io/BBHx/user/like.html#bbhx.likelihood.HeterodynedLikelihood).
First, a **cheap self-contained illustration** of *why* it works &mdash; no waveform
generation, just two nearby analytic chirps &mdash; then a **sketch** of the real
BBHx call.

Useful documentation:
* [`HeterodynedLikelihood`](https://lisa-analysis-tools.github.io/BBHx/user/like.html#bbhx.likelihood.HeterodynedLikelihood) (`get_ll`, `init_heterodyne_info`; BBHx)
* Informational notebook: see [`07`](../../07_ErynSmallToLarge.ipynb) and the fast-likelihood Question in [`X6` &sect; Task 5](X6_GalacticBinaryMCMC.ipynb)

And the real thing for MBHBs, mirroring the old Tutorial 4 (a **sketch** &mdash;
we do not build the full BBHx generator here to stay inside the memory/time budget).
`HeterodynedLikelihood` wraps a frequency-domain BBHx waveform generator; you build it
once with the data and a reference parameter vector, then `get_ll` is your fast
log-likelihood inside the sampler:

```python
from bbhx.likelihood import HeterodynedLikelihood

length_f_het = 128        # number of relative-binning knots (hundreds, not millions)
het_like = HeterodynedLikelihood(
    bbh_wave_gen,         # a bbhx frequency-domain MBHB generator
    freqs,                # the fine data-frequency grid
    data,                 # the injected FD data (A, E, T or X, Y, Z)
    reference_params,     # a template near the truth -> defines h_0(f)
    length_f_het,
)

# drop-in fast log-likelihood; refresh the reference if the chain wanders far:
ll = het_like.get_ll(trial_params)
het_like.init_heterodyne_info(new_reference_params)   # re-seed h_0 when needed
```

That single swap &mdash; `ac.eryn_likelihood_function` &rarr; `het_like.get_ll` &mdash;
is what turns the minutes-long toy run above into a production MBHB analysis.

### Question: why does heterodyning matter most for long, loud signals?

Relative binning helps every source a little, but it is *transformational*
for MBHBs (and other long, loud signals). Why &mdash; what is it about high SNR and a
long, broadband chirp that makes the speed-up so large, and where does the trick
break down?

*Discussion.* The cost of a *direct* likelihood is $O(N_\text{bins})$, and
$N_\text{bins}$ grows with both the **observation length** and the **bandwidth** of
the signal: a loud MBHB sweeps across a wide frequency range over days, so its data
occupy a huge number of bins (millions for a full run), and *every* one is rescored on
*every* likelihood call. Heterodyning replaces that with a few hundred knots, so the
saving &mdash; the ratio $N_\text{bins}/N_\text{knots}$ &mdash; is largest exactly
when the signal is longest and broadest, i.e. for the loudest MBHBs. High SNR helps in
a second way: it makes the posterior **narrow**, so the chain stays close to any
reference template and $r(f)=h/h_0$ really is smooth over the whole run, which is the
condition relative binning needs. The flip side is *where it breaks*: if a trial
template is **far** from the reference (early in a search, a badly chosen $h_0$, or a
multi-modal posterior), $r(f)$ is no longer smooth and the coarse-grid approximation
is wrong &mdash; which is why production pipelines **re-seed the reference**
(`init_heterodyne_info`) as the chain moves and only lean on heterodyning once they are
near a mode. It is the same lesson as the band-limited GB likelihood in
[`X6`](X6_GalacticBinaryMCMC.ipynb): exploit the fact that a source's information lives
in a small, structured part of the data, and never pay for the rest.

*Going deeper.* For a quick SNR **forecast** without any of this machinery,
`lisatools.sources.bbh` also ships
[`BBHSNRWaveform`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/main.html)
&mdash; a sparse, log-spaced BBHx frequency-domain waveform that reports the **A, E, T**
SNR directly (the one place we quote AET rather than XYZ). It is handy for
back-of-the-envelope detectability, but it is *not* the global-fit residual template:
the fit scores the `PhenomTHMTDIWaveform` XYZ channels you used above.

### Where this goes next

You have now run the full **source-MCMC skeleton** on a massive black-hole binary:
a `PhenomTHMTDIWaveform` injection &rarr; the LISA response & TDI &rarr; a loud SNR
&rarr; a band-limited `signal_gen` &rarr; an `AnalysisContainer` likelihood &rarr; a
tempered sampler and a corner plot &rarr; and the **heterodyned** fast likelihood that
makes a real run affordable. It is the *same* skeleton as
[`X6`](X6_GalacticBinaryMCMC.ipynb)'s galactic binary and
[`X2`](X2_EMRIResponseMCMC.ipynb)'s EMRI &mdash; only the waveform changed. The
distance from here to a real MBHB analysis is not a new idea about the sampler but
*scale*: all ~11 parameters, the heterodyned likelihood as the default, GPU waveforms,
tuned proposals, and a proper search to seed it (informational notebooks
[`07`](../../07_ErynSmallToLarge.ipynb) and
[`03`](../../03_StockGlobalFitsInDepth.ipynb)).